After training the attentive classifier with verb, manipulated and affected prediction objective. We don't need the classes output logits, we only need the feature probing from attentive pooler. Hence the final feature is the output feature coming from attentive pooler.

# Load classifier with weights

In [ ]:
import yaml
import json
import torch
from pprint import pprint
from glob import glob

# from src.classifiers import init_classifier

# for debug
import os
from src.classifiers import AttentiveClassifier

with open('data/annotations/annotation_all.json', 'r') as f:
    annota = json.load(f)
typeinfo = annota['type_info']
NUM_CLASSES = {k: v['num_classes'] for k,v in typeinfo.items()}
print(NUM_CLASSES)

with open('configs/feature_extraction_vjepa2.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
cls_cfg = cfg['classifier']
assert cls_cfg is not None, 'Classifier config is emtpy!'

pprint(cfg)


def init_classifier(
    embed_dim: int,
    num_heads: int,
    num_blocks: int,
    num_verb_classes: int,
    num_mani_classes: int,
    num_affect_classes: int,
    checkpoint: str=None,
):
    classifier = AttentiveClassifier(
        embed_dim=embed_dim,
        num_heads=num_heads,
        depth=num_blocks,
        num_verb_classes=num_verb_classes,
        num_manipulated_classes=num_mani_classes,
        num_affected_classes=num_affect_classes,
    )
    
    if checkpoint is not None:
        if os.path.isdir(checkpoint):
            ckpt_paths = sorted(glob(os.path.join(checkpoint, "*.pt")))
            ckpt = ckpt_paths[-1]
        elif os.path.isfile(checkpoint):
            ckpt = checkpoint
        ckpt = torch.load(ckpt, map_location="cpu")
        classifier.load_state_dict(ckpt["classifier"])
        del ckpt

    return classifier

classifer = init_classifier(
    # checkpoint='ckpt/epoch1.pt',
    num_verb_classes=NUM_CLASSES['verb'],
    num_mani_classes=NUM_CLASSES['manipulated'],
    num_affect_classes=NUM_CLASSES['affected'],
    **cls_cfg
)



{'verb': 10, 'manipulated': 35, 'affected': 36, 'atomic_operation': 244, 'hand': 2}
{'classifier': {'embed_dim': 1024, 'num_blocks': 4, 'num_heads': 16},
 'dataset': {'allow_clip_overlap': True,
             'frames_per_clip': 16,
             'json_file': 'data/annotations/annotation_all.json',
             'label2id_dir': 'data/annotations',
             'random_view': False,
             'sampling_rate': 1,
             'video_dir': ['data/FineBio/03 finebio_videos_fpv_all_w640',
                           'data/FineBio/10 '
                           'finebio_videos_tpv_all_w640/finebio_videos_w640']},
 'init_rand_seed': 2708,
 'opt': {'lr': 0.000425, 'warmup': 5, 'wd': 0.04}}


In [ ]:
classifer.pooler

<bound method Module.modules of AttentiveClassifier(
  (pooler): AttentivePooler(
    (cross_attention_block): CrossAttentionBlock(
      (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (xattn): CrossAttention(
        (q): Linear(in_features=1024, out_features=1024, bias=True)
        (kv): Linear(in_features=1024, out_features=2048, bias=True)
      )
      (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (blocks): ModuleList(
      (0-2): 3 x Block(
        (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Li

# Load V-JEPA 2

In [ ]:
from src.models import init_vjepa2

model, processor = init_vjepa2()

# Extract loop

In [1]:
import os
import yaml
import torch
import numpy as np
from pprint import pprint
from tqdm import tqdm

from src.data_utils import get_video_paths, decode_video_to_clips

In [ ]:
# === prepare save dir ===
save_dir = 'feats'
if not os.path.isdir(save_dir):
    os.mkdir(save_dir)

# === save config ===
with open(os.path.join(save_dir, "feature_extraction_config.txt"), "w") as f:
    yaml.dump(cfg, f)

# === init data ===
video_path_list = get_video_paths(
    data_dir=cfg["video_dir"], 
    # parameters for resuming feature extraction
    target_view_id="T0", 
    min_par_id=0)


In [ ]:
# === iterate videos ===
device = model.device
decode_cfg = cfg['decode_kwargs']
for path in tqdm(video_path_list):
    clips = decode_video_to_clips(
        video_path=path,
        frames_per_clip=decode_cfg["frames_per_clip"],
        clip_stride=decode_cfg["clip_stride"],
        sampling_rate=decode_cfg["sampling_rate"]
    )

    feat_list = []
    for clip in tqdm(clips):
        buffer = clip["buffer"] # T,H,W,3
        buffer = processor(
            buffer,
            return_tensors="pt",
            do_resize=False
        )["pixel_values_videos"].float().to(device, non_blocking=True) # B,T,3,H,W

        with torch.no_grad():
            feat = model(buffer).last_hidden_state # ['last_hidden_state', 'masked_hidden_state', 'predictor_output']
            feat = classifer.pooler(feat) # B, 3, D
        
        feat_list.append(feat.data.cpu().numpy())
        num_clips = clip["num_clips"]
    
    assert len(feat_list) == num_clips, "Invalid feature length!"

    # === save video feat ===
    video_id = os.path.splitext(os.path.basename(path))[0]
    feat_path = video_id + ".npy"
    np.save(os.path.join(save_dir, feat_path), feat_list)
    print(f"{video_id}: num feat = ", len(feat_list))